<a href="https://colab.research.google.com/github/MattManson/lastfm-music-pipeline/blob/main/lastfm_medallion_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import os
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages io.delta:delta-spark_2.12:3.3.0 pyspark-shell'
print("Delta environment configured")

Delta environment configured


In [10]:
!pip install delta-spark==3.3.0

In [11]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("lastfm-pipeline") \
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .master("local[*]") \
    .getOrCreate()

print(spark.version)

3.5.8


In [12]:
from IPython.core.display import json
import requests
from google.colab import userdata

API_KEY = userdata.get('LASTFM_API_KEY')

BASE_URL = "http://ws.audioscrobbler.com/2.0/"

def get_top_tracks(country, limit=50):
    """
    Fetch top tracks for a given country from Last.fm
    country: country name as a string e.g. 'united kingdom'
    limit: how many tracks to return, max 50 per page
    """
    url = f"http://ws.audioscrobbler.com/2.0/?method=geo.gettoptracks&country={country}&limit={limit}&api_key={API_KEY}&format=json"

    response = requests.get(url)
    print(response.status_code)
    data = response.json()
    return data

result = get_top_tracks("united kingdom", 5)
print(result)

200
{'tracks': {'track': [{'name': 'Stateside + Zara Larsson', 'duration': '176', 'listeners': '32616', 'mbid': 'ffbf7862-2476-4164-ac32-f5904ccefe0f', 'url': 'https://www.last.fm/music/PinkPantheress/_/Stateside+%252B+Zara+Larsson', 'streamable': {'#text': '0', 'fulltrack': '0'}, 'artist': {'name': 'PinkPantheress', 'mbid': '7441014f-f8f5-494f-81db-ff166fbc078d', 'url': 'https://www.last.fm/music/PinkPantheress'}, 'image': [{'#text': 'https://lastfm.freetls.fastly.net/i/u/34s/2a96cbd8b46e442fc41c2b86b821562f.png', 'size': 'small'}, {'#text': 'https://lastfm.freetls.fastly.net/i/u/64s/2a96cbd8b46e442fc41c2b86b821562f.png', 'size': 'medium'}, {'#text': 'https://lastfm.freetls.fastly.net/i/u/174s/2a96cbd8b46e442fc41c2b86b821562f.png', 'size': 'large'}, {'#text': 'https://lastfm.freetls.fastly.net/i/u/300x300/2a96cbd8b46e442fc41c2b86b821562f.png', 'size': 'extralarge'}], '@attr': {'rank': '0'}}, {'name': 'American Girls', 'duration': '213', 'listeners': '19099', 'mbid': '2c85fe70-3c0e-4b4

In [13]:
BASE_PATH = "/tmp/lastfm_pipeline"
BRONZE_PATH = f"{BASE_PATH}/bronze"
SILVER_PATH = f"{BASE_PATH}/silver"
GOLD_PATH = f"{BASE_PATH}/gold"

print(f"Bronze: {BRONZE_PATH}")
print(f"Silver: {SILVER_PATH}")
print(f"Gold: {GOLD_PATH}")

Bronze: /tmp/lastfm_pipeline/bronze
Silver: /tmp/lastfm_pipeline/silver
Gold: /tmp/lastfm_pipeline/gold


In [14]:
import json
from datetime import datetime

def store_bronze(raw_data, source):
    """
    Convert raw API response to a PySpark DataFrame
    representing the bronze layer - raw data with ingestion timestamp

    raw_data: the raw dictionary returned from the API
    source: a label identifying where the data came from
    """

    # Get current timestamp to record when we ingested this data
    ingested_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # Convert the entire raw response to a JSON string
    # We store it as one big string - that's intentional for bronze
    raw_json_string = json.dumps(raw_data)

    # Create a single row DataFrame with the raw data and metadata
    bronze_data = [(source, ingested_at, raw_json_string)]
    bronze_columns = ["source", "ingested_at", "raw_json"]

    bronze_df = spark.createDataFrame(bronze_data, bronze_columns)
    return bronze_df

# Test it
bronze_df = store_bronze(result, "lastfm_geo_toptracks_uk")
bronze_df.show(truncate=50)

+-----------------------+-------------------+--------------------------------------------------+
|                 source|        ingested_at|                                          raw_json|
+-----------------------+-------------------+--------------------------------------------------+
|lastfm_geo_toptracks_uk|2026-03-23 15:35:04|{"tracks": {"track": [{"name": "Stateside + Zar...|
+-----------------------+-------------------+--------------------------------------------------+



In [15]:
from datetime import datetime
import json

def write_bronze(raw_data, source):
    """
    Write raw API response to the bronze Delta table.
    Bronze stores data exactly as received with a timestamp.
    We append each run rather than overwrite - building up history.
    """
    ingested_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    raw_json_string = json.dumps(raw_data)

    bronze_data = [(source, ingested_at, raw_json_string)]
    bronze_columns = ["source", "ingested_at", "raw_json"]

    bronze_df = spark.createDataFrame(bronze_data, bronze_columns)

    # Write to Delta format - append means we add to existing data
    # rather than overwriting it each run
    bronze_df.write \
        .format("delta") \
        .mode("append") \
        .save(f"{BRONZE_PATH}/raw_api_responses")

    print(f"Bronze write complete: {source} at {ingested_at}")
    return bronze_df

# Test it
write_bronze(result, "lastfm_geo_toptracks_uk")

Bronze write complete: lastfm_geo_toptracks_uk at 2026-03-23 15:35:05


DataFrame[source: string, ingested_at: string, raw_json: string]

In [16]:
bronze_df_check = spark.read.format("delta").load(f"{BRONZE_PATH}/raw_api_responses")
bronze_df_check.show(truncate=50)

+-----------------------+-------------------+--------------------------------------------------+
|                 source|        ingested_at|                                          raw_json|
+-----------------------+-------------------+--------------------------------------------------+
|lastfm_geo_toptracks_uk|2026-03-23 15:35:05|{"tracks": {"track": [{"name": "Stateside + Zar...|
|lastfm_geo_toptracks_uk|2026-03-23 15:34:38|{"tracks": {"track": [{"name": "Stateside + Zar...|
+-----------------------+-------------------+--------------------------------------------------+



In [30]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

def parse_tracks(raw_data, snapshot_date):
    """
    Extract track records from raw Last.fm API response.
    Takes the nested JSON and returns a flat list of tuples.

    raw_data: the raw dictionary from the API
    snapshot_date: the date this data was captured
    """

    tracks = raw_data["tracks"]["track"]

    parsed = []

    for track in tracks:
      parsed.append((
          track["name"],
          track["artist"]["name"],
          int(track["duration"]),
          int(track["listeners"]),
          int(track["@attr"]["rank"]),
          snapshot_date
      ))
    return parsed

track_schema = StructType([
    StructField("track_name", StringType(), True),
    StructField("artist_name", StringType(), True),
    StructField("duration_seconds", IntegerType(), True),
    StructField("listeners", IntegerType(), True),
    StructField("chart_rank", IntegerType(), True),
    StructField("snapshot_date", StringType(), True)
])

# Test the parsing
parsed_tracks = parse_tracks(result, "2026-03-23")
print(f"Parsed {len(parsed_tracks)} tracks")
print(parsed_tracks[0])

Parsed 5 tracks
('Stateside + Zara Larsson', 'PinkPantheress', 176, 32616, 0, '2026-03-23')


In [31]:
def write_silver_tracks(parsed_tracks, schema):

    """
    Write parsed track data to the silver Delta table.
    Silver is clean, typed and structured, and ready for analysis
    """

    silver_df = spark.createDataFrame(parsed_tracks, schema)

    silver_df.write \
        .format("delta") \
        .mode("append") \
        .save(f"{SILVER_PATH}/tracks")

    print(f"Silver write complete: {len(parsed_tracks)} tracks written")
    return silver_df
silver_df = write_silver_tracks(parsed_tracks, track_schema)
silver_df.show()

Silver write complete: 5 tracks written
+--------------------+--------------+----------------+---------+----------+-------------+
|          track_name|   artist_name|duration_seconds|listeners|chart_rank|snapshot_date|
+--------------------+--------------+----------------+---------+----------+-------------+
|Stateside + Zara ...|PinkPantheress|             176|    32616|         0|   2026-03-23|
|      American Girls|  Harry Styles|             213|    19099|         1|   2026-03-23|
|Rein Me In (with ...|    Sam Fender|               0|    15679|         2|   2026-03-23|
|            Babydoll|  Dominic Fike|              97|    15559|         3|   2026-03-23|
|          Man I Need|   Olivia Dean|             184|    14763|         4|   2026-03-23|
+--------------------+--------------+----------------+---------+----------+-------------+



In [33]:
import time

def get_artist_tags(artist_name):
    """
    Fetch genre tags for a single artist from last_fm
    returns a list of (artist_name, tag_name, tag_count) tuples
    """
    url = f"http://ws.audioscrobbler.com/2.0/?method=artist.gettoptags&artist={artist_name}&api_key={API_KEY}&format=json"

    response = requests.get(url)
    data = response.json()

    # Check the response has tags - some artists return an error
    if "toptags" not in data:
        print(f"No tags found for {artist_name}")
        return[]

    tags = data["toptags"]["tag"]

    parsed_tags = []
    for tag in tags:
        parsed_tags.append((
            artist_name,
            tag["name"],
            int(tag["count"])
        ))
    return parsed_tags

def get_all_artist_tags(parsed_tracks):
    """
    Fetch tags for every unique artist in our tracks list
    parsed_tracks: the list of track tuples from parse_tracks
    """

    artists = list(set([track[1] for track in parsed_tracks]))

    all_tags = []
    for artist in artists:
        print(f"fetching tags for {artist}...")
        tags = get_artist_tags(artist)
        all_tags.extend(tags)
        time.sleep(0.2) #200ms delay between calls to not brick our api terms

    return all_tags

all_tags = get_all_artist_tags(parsed_tracks)
print(f"\nTotal tag records: {len(all_tags)}")
print(all_tags[0])

fetching tags for Dominic Fike...
fetching tags for Harry Styles...
fetching tags for Sam Fender...
fetching tags for PinkPantheress...
fetching tags for Olivia Dean...

Total tag records: 50
('Dominic Fike', 'indie', 100)


In [35]:
from pyspark.sql.types import StructField, StructType, StringType, IntegerType

artist_tag_schema = StructType([
    StructField("artist_name", StringType(), True),
    StructField("tag_name", StringType(), True),
    StructField("tag_count", IntegerType(), True)
])

def write_silver_tags(all_tags, schema):
    """
    Write artist tag data to the silver Delta table.
    """
    tags_df = spark.createDataFrame(all_tags, schema)

    tags_df.write \
        .format("delta") \
        .mode("append") \
        .save(f"{SILVER_PATH}/artist_tags")

    print(f"Silver tags write complete: {len(all_tags)} tag records written")
    return tags_df

tags_df = write_silver_tags(all_tags, artist_tag_schema)
tags_df.show()

Silver tags write complete: 50 tag records written
+------------+---------------+---------+
| artist_name|       tag_name|tag_count|
+------------+---------------+---------+
|Dominic Fike|          indie|      100|
|Dominic Fike|      indie pop|       44|
|Dominic Fike|            pop|       20|
|Dominic Fike|    alternative|       13|
|Dominic Fike|       american|        7|
|Dominic Fike|alternative pop|        4|
|Dominic Fike|        Hip-Hop|        4|
|Dominic Fike|            USA|        1|
|Dominic Fike|   my top songs|        1|
|Dominic Fike|        florida|        1|
|Harry Styles|            pop|      100|
|Harry Styles|           rock|       77|
|Harry Styles|       pop rock|       43|
|Harry Styles|    alternative|       31|
|Harry Styles|        british|       13|
|Harry Styles|      soft rock|        7|
|Harry Styles|          indie|        7|
|Harry Styles|           folk|        2|
|Harry Styles|  one direction|        2|
|Harry Styles|       bisexual|        2|
+-----